####  Run this cell to set up and start your interactive session.


In [1]:
%stop_session
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5
%additional_python_modules TextBlob, gensim, scikit-learn

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, count

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.7 
There is no current session.
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Additional python modules to be included:
TextBlob
gensim
scikit-learn
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 090f4fab-f5a1-4ccc-a4d2-1f32c0bdd279
Applying the following default arguments:
--glue_kernel_version 1.0.7
--enable-glue-datacatalog true
--additional-python-modules Text

In [2]:
dynamo_table = "Peliculas_S3D2_xideral"

dyf = glueContext.create_dynamic_frame.from_options(
    connection_type="dynamodb",
    connection_options={"dynamodb.input.tableName": dynamo_table,
        "dynamodb.throughput.read.percent": "1.0",
        "dynamodb.splits": "100"
    }
)

In [3]:
df = dyf.toDF()
df.show(3)

+--------------------+-----------+---------+-------------+--------------------+-----------+
|          fecha_hora|   duracion|     sala|clasificacion|              nombre|pelicula_id|
+--------------------+-----------+---------+-------------+--------------------+-----------+
| 2025-02-07 16:51:11|{114, NULL}|{4, NULL}|        PG-13|              Barbie|         63|
| 2025-02-14 12:17:55|{169, NULL}|{5, NULL}|            R|         John Wick 4|         64|
|2025-02-05T10:12:...|{NULL, 148}|{NULL, 7}|            B|Spider-Man: No Wa...|         36|
+--------------------+-----------+---------+-------------+--------------------+-----------+
only showing top 3 rows

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


In [4]:
from pyspark.sql.functions import col, count

In [5]:
# Contamos las peliculas mas anunciadas en funciones
df_titles_count = df.groupBy("nombre").agg(count("*").alias("funciones"))

# Tomamos las primeras 10
df_titles_count = df_titles_count.orderBy(col("funciones").desc()).limit(10)
df_titles_count.show(10)

+--------------------+---------+
|              nombre|funciones|
+--------------------+---------+
|Spider-Man: No Wa...|        4|
|         Oppenheimer|        2|
|            Conclave|        2|
|           Inception|        2|
|           Nosferatu|        2|
|        La Sustancia|        2|
|               Elvis|        2|
|   Johanne Sacreblue|        2|
|Avatar: The Way o...|        2|
|        Interstellar|        2|
+--------------------+---------+


In [6]:
# Contamos las clasificaciones mas anunciadas en funciones
df_titles_count = df.groupBy("clasificacion").agg(count("*").alias("funciones"))

# Tomamos las primeras 5
df_titles_count = df_titles_count.orderBy(col("funciones").desc()).limit(5)
df_titles_count.show(5)

+-------------+---------+
|clasificacion|funciones|
+-------------+---------+
|        PG-13|       22|
|            R|       12|
|            B|       11|
|            A|       11|
|            C|        7|
+-------------+---------+


In [7]:
df_categories = df.select("clasificacion").distinct()
df_categories.show()

+-------------+
|clasificacion|
+-------------+
|            B|
|            R|
|            D|
|        PG-13|
|            C|
|            A|
|          B15|
|           AA|
|           PG|
+-------------+


In [8]:
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = api.load("glove-twitter-25")  # Modelo ligero
positive_words = ["love", "happy", "amazing", "wonderful", "joy", "peace"]
negative_words = ["kill", "death", "dark", "revenge", "fear", "sad", "horror"]

def get_sentiment(title, category):
    
    words = title.lower().split()
    
    # Filtrar palabras que están en el vocabulario de Word2Vec
    words = [word for word in words if word in model]

    if not words:
        return "Neutral"

    title_vector = np.mean([model[word] for word in words], axis=0).reshape(1, -1)
    positive_vectors = np.array([model[word] for word in positive_words if word in model])
    negative_vectors = np.array([model[word] for word in negative_words if word in model])

    # Calcular similitud coseno con la media
    pos_score = np.mean(cosine_similarity(title_vector, positive_vectors))
    neg_score = np.mean(cosine_similarity(title_vector, negative_vectors))

    # Obtener el peso basado en la clasificación
    category_weight = category_dict.get(category)

    # Ajustar los puntajes con el peso de la clasificación
    if category_weight < 0:
        pos_score *= -0.5*category_weight
        neg_score *= -1*category_weight
    if category_weight >= 0:
        pos_score *= category_weight
        neg_score *= 0.5*category_weight

    # Comparacion Final
    if pos_score > neg_score:
        return f"{pos_score:.2f} , Comedia / Aventura"
    elif neg_score > pos_score:
        return f"{neg_score:.2f} , Terror / Suspenso"
    else:
        return f"{pos_score:.2f} , Drama"


[==================================================] 100.0% 104.8/104.8MB downloaded


In [9]:
import numpy as np

category_list = ["AA", "A", "B", "PG", "PG-13", "B15", "C", "D", "R"]

# Definir los valores de inicio y final
start = 2  
end = -2  

# Generar una escala lineal con `linspace`
num_points = len(category_list)
lin_scale = np.linspace(start, end, num=num_points)

# Evitar que haya exactamente un 0
epsilon = 0.05  # Pequeño ajuste
lin_scale = np.where(lin_scale == 0, epsilon, lin_scale)

# Diccionario de categoria-peso
category_dict = {category: value for category, value in zip(category_list, lin_scale)}


<stdin>:7: RuntimeWarning: invalid value encountered in log10


In [10]:
from textblob import TextBlob

"""
def get_sentiment(text, category):
    category_list = ["AA", "A", "B","PG","PG-13","B15", "C", "D","R"]
    start = 1
    end = 1
    num_points = len(category_list)
    log_scale = np.logspace(np.log10(start), np.log10(end), num=num_points)
    
    # Se crea el diccionario asignando los valores de la escala logarítmica

    category_dict = {category: value for category, value in zip(category_list, log_scale)}
    
    score = TextBlob(text).sentiment.polarity*category_dict[category]
    if score > 0.25:
        return str(score)+ " , " + "Comedia / Aventura"
    elif score < -0.25:
        return str(score)+ " , " + "Terror / Suspenso"
    else:
        return str(score)+ " , " + "Drama"
"""
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

sentiment_udf = udf(get_sentiment, StringType())
df = df.withColumn("genero", sentiment_udf(df["nombre"],df["clasificacion"]))

df.select("nombre", "genero").show()


+--------------------+--------------------+
|              nombre|              genero|
+--------------------+--------------------+
|              Barbie|0.56 , Comedia / ...|
|         John Wick 4|0.56 , Comedia / ...|
|Spider-Man: No Wa...|0.73 , Comedia / ...|
|             Titanic|0.60 , Terror / S...|
|           Nosferatu|0.12 , Terror / S...|
|           Nosferatu|0.12 , Terror / S...|
|Zack Snyder's Jus...|0.64 , Terror / S...|
|Spider-Man: No Wa...|0.73 , Comedia / ...|
|          The Matrix|0.76 , Terror / S...|
|Avatar: The Way o...|0.79 , Terror / S...|
|         El Rey León|0.23 , Terror / S...|
|Batman: El caball...|0.21 , Comedia / ...|
|        Forrest Gump|0.52 , Terror / S...|
|        Training Day|0.76 , Comedia / ...|
|   Top Gun: Maverick|0.57 , Comedia / ...|
|              Avatar|0.37 , Comedia / ...|
|       Blood Diamond|0.68 , Comedia / ...|
|               Joker|0.64 , Terror / S...|
| Mi Villano Favorito|0.20 , Comedia / ...|
|        La Sustancia|0.19 , Ter

In [11]:
from awsglue.dynamicframe import DynamicFrame
dyf = DynamicFrame.fromDF(df, glueContext, "dynamic")
glueContext.write_dynamic_frame.from_options(
        frame=dyf,
        connection_type="dynamodb",
        connection_options={"dynamodb.output.tableName": "josue_analysis_movies"}
)

In [23]:
dyf = glueContext.create_dynamic_frame.from_options(
    connection_type="dynamodb",
    connection_options={"dynamodb.input.tableName": "josue_analysis_movies",
        "dynamodb.throughput.read.percent": "1.0",
        "dynamodb.splits": "100"
    }
)
dyf.show(3)

{"fecha_hora": "2025-02-07 16:51:11", "genero": "0.56 , Comedia / Aventura", "duracion": {"string": null, "long": 114}, "sala": {"string": null, "long": 4}, "clasificacion": "PG-13", "nombre": "Barbie", "pelicula_id": "63"}
{"fecha_hora": "2025-02-14 12:17:55", "genero": "0.28 , Comedia / Aventura", "duracion": {"string": null, "long": 169}, "sala": {"string": null, "long": 5}, "clasificacion": "R", "nombre": "John Wick 4", "pelicula_id": "64"}
{"fecha_hora": "2025-02-05T10:12:36.075352", "genero": "1.03 , Comedia / Aventura", "duracion": {"string": "148", "long": null}, "sala": {"string": "7", "long": null}, "clasificacion": "B", "nombre": "Spider-Man: No Way Home", "pelicula_id": "36"}
